In [ ]:
import yaml
import numpy as np
import matplotlib.pyplot as plt
import h5py
from fastnoise import NoiseFile
from tqdm import tqdm

In [ ]:
hf_noise = h5py.File('/Users/michield/Documents/PhD/Code/KARAOCE/kid_test/fastreadout_noise.00174.h5', 'r')
noise = NoiseFile('/Users/michield/Documents/PhD/Code/KARAOCE/kid_test/fastreadout_noise.00174.h5')

In [ ]:
n_pixels = noise.numPixels

sigma = np.zeros(n_pixels)
thresholds = []

for i, pixel_dacbin in tqdm(enumerate(noise.allPixels)):

	pixel_noise = noise[pixel_dacbin].allPhaseData * (-1)

	# unwrap
	pixel_noise = np.unwrap(pixel_noise, discont=None, axis=-1, period=2*np.pi)
	if np.average(pixel_noise) > np.pi:
			pixel_noise -= 2*np.pi
	elif np.average(pixel_noise) < -np.pi:
		pixel_noise += 2*np.pi

	# sigma noise
	sigma[i] = np.std(pixel_noise)

	# trigger threshold
	threshold_level = 7
	thresholds.append({'pixel_number': int(noise[pixel_dacbin].pixelNr), 'threshold': float(np.clip(-threshold_level * sigma[i], -np.pi, np.pi))})

In [ ]:
10**0.2

In [ ]:
for i, pixel_dacbin in tqdm(enumerate(noise.allPixels)):
	print(pixel_dacbin, noise[pixel_dacbin].pixelNr)

In [ ]:
%matplotlib widget

In [ ]:
test = -168808
print(test, noise[test].pixelNr)

pixel_noise = noise[test].allPhaseData * (-1)

pixel_noise_unwrapped = np.unwrap(pixel_noise, discont=None, axis=-1, period=2*np.pi)
if np.average(pixel_noise_unwrapped) > np.pi:
		pixel_noise_unwrapped -= 2*np.pi
elif np.average(pixel_noise_unwrapped) < -np.pi:
	pixel_noise_unwrapped += 2*np.pi

print(np.std(pixel_noise_unwrapped))
print(np.std(pixel_noise_unwrapped) * 7)

plt.figure()
plt.plot(pixel_noise, marker='*', label='data')
plt.plot(pixel_noise_unwrapped, label='unwrapped')
plt.legend()

In [ ]:
print(thresholds)

In [ ]:
# Writing the data to a YAML file
with open('trigger_thresholds.yaml', 'w') as file:
    yaml.dump(thresholds, file)

In [ ]:
# Reading data from the YAML file
with open('trigger_thresholds.yaml', 'r') as file:
    loaded_data = yaml.safe_load(file)

print("Data read from 'data.yaml':")
print(loaded_data)